# 🔊 Notebook 07 — Audio Deepfake Detection (CRNN)
Person 2

In [6]:
!pip install torch torchvision tqdm scikit-learn librosa -q

In [7]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import accuracy_score, roc_auc_score

# ── Debug: inspect what's mounted ──────────────────────────────────────
print("MyDrive contents:", os.listdir("/content/drive/MyDrive"))

# ── After adding shortcut, the path will be under MyDrive ──────────────
BASE_DIR = "/content/drive/MyDrive/deepfake-project"

if not os.path.exists(BASE_DIR):
    raise FileNotFoundError(
        f"❌ '{BASE_DIR}' not found.\n"
        "➡ Go to Google Drive → Shared with me → right-click deepfake-project "
        "→ Organize → Add shortcut → My Drive"
    )

print(f"✅ Found project at: {BASE_DIR}")
print("Contents:", os.listdir(BASE_DIR))

# ── Note: folder on drive is 'spectrograms_npy' (check exact casing) ───
SPEC_DIR  = os.path.join(BASE_DIR, "data/spectrograms_npy")
MODEL_DIR = os.path.join(BASE_DIR, "models")
os.makedirs(MODEL_DIR, exist_ok=True)

# Verify SPEC_DIR exists
if not os.path.exists(SPEC_DIR):
    print("data/ contents:", os.listdir(os.path.join(BASE_DIR, "data")))
    raise FileNotFoundError(f"❌ SPEC_DIR not found: {SPEC_DIR}")

DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 64; EPOCHS = 20; LR = 1e-3; SPEC_SIZE = 128; SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print(f"✅ Device: {DEVICE}")

Mounted at /content/drive
MyDrive contents: ['Getting started.pdf', 'Classroom', '2.pdf', '3.pdf', '4.pdf', '5.pdf', '1.pdf', 'WhatsApp Image 2023-05-15 at 16.24.57.gdoc', 'Photo from E. Varshith (1).gdoc', 'Photo from E. Varshith.gdoc', 'Photo from E. Varshith (1)', 'Photo from E. Varshith', 'Colab Notebooks', 'My Resume (2).pdf', 'idCard.jpg', 'My_Resume.pdf', '126 (1).pdf', 'Resume (blue).pdf', '50dayLeetcode.png', 'Nubra_Assignment.pdf', 'Latest_Resume.pdf', 'Varshith Eturu_22BCE7295_Healthcare Symptom Checker.docx', 'Rentillgence : Capstone Project Report.gdoc', 'titled document.gdoc', 'DT20257185951_Application_Form.pdf', 'NewLatest_Resume.pdf', 'RealLatest_Resume.pdf', 'TCS_Resume.pdf', 'Copy of MaxOfJob Job Tracker Template.gsheet', 'Untitled document.gdoc', 'Job Hunting.gsheet', 'testing', 'deepfake-project']
✅ Found project at: /content/drive/MyDrive/deepfake-project
Contents: ['notebooks', 'data', 'Doc.gdoc', 'models', 'Untitled presentation.gslides', 'Explainable Multimodal

In [8]:
# ✅ Dataset
class SpectrogramDataset(Dataset):
    def __init__(self, files, labels, augment=False):
        self.files = files; self.labels = labels; self.augment = augment
    def __len__(self): return len(self.files)
    def __getitem__(self, idx):
        spec = np.load(self.files[idx]).astype(np.float32)
        spec = np.expand_dims(spec, 0)
        if self.augment and random.random() < 0.4:
            f0 = random.randint(0, SPEC_SIZE//4)
            spec[0, f0:f0+random.randint(5,20), :] = 0.0
        return torch.tensor(spec), torch.tensor(self.labels[idx], dtype=torch.float32)

def load_split(spec_dir):
    files, labels = [], []
    for split, lbl in [("fake_with_audio",1),("real_with_audio",0)]:
        d = os.path.join(spec_dir, split)
        if not os.path.isdir(d): print(f"[SKIP] {d}"); continue
        for fp in Path(d).glob("*.npy"):
            files.append(str(fp)); labels.append(lbl)
    return files, labels

all_files, all_labels = load_split(SPEC_DIR)
combined = list(zip(all_files,all_labels)); random.shuffle(combined)
all_files, all_labels = zip(*combined)
n_val = int(len(all_files)*0.15); n_test = int(len(all_files)*0.10)
tr_f,tr_l = list(all_files[n_val+n_test:]),list(all_labels[n_val+n_test:])
va_f,va_l = list(all_files[:n_val]),list(all_labels[:n_val])
te_f,te_l = list(all_files[n_val:n_val+n_test]),list(all_labels[n_val:n_val+n_test])
tr_dl = DataLoader(SpectrogramDataset(tr_f,tr_l,augment=True),batch_size=BATCH_SIZE,shuffle=True,num_workers=2)
va_dl = DataLoader(SpectrogramDataset(va_f,va_l),batch_size=BATCH_SIZE,num_workers=2)
te_dl = DataLoader(SpectrogramDataset(te_f,te_l),batch_size=BATCH_SIZE,num_workers=2)
print(f"✅ Train: {len(tr_f)}, Val: {len(va_f)}, Test: {len(te_f)}")

✅ Train: 782, Val: 156, Test: 104


In [9]:
# ✅ CRNN Model
class AudioCRNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64,128,3,padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.gru = nn.GRU(128*16, 128, batch_first=True, bidirectional=True)
        self.fc  = nn.Sequential(nn.Dropout(0.5),nn.Linear(256,64),nn.ReLU(),nn.Linear(64,1))
    def forward(self, x):
        feat = self.cnn(x); B,C,H,W = feat.shape
        feat = feat.permute(0,3,1,2).reshape(B,W,C*H)
        out,_ = self.gru(feat)
        return self.fc(out[:,-1,:]).squeeze(1)

model     = AudioCRNN().to(DEVICE)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.5)
print("✅ CRNN model ready.")

✅ CRNN model ready.


In [10]:
# ✅ Training
best_auc = 0.0
for epoch in range(1, EPOCHS+1):
    model.train(); tr_loss = 0
    for spec, lbl in tqdm(tr_dl, desc=f"Epoch {epoch}/{EPOCHS}"):
        spec, lbl = spec.to(DEVICE), lbl.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(spec), lbl)
        loss.backward(); optimizer.step(); tr_loss += loss.item()
    scheduler.step()
    model.eval(); probs, lbls_all = [], []
    with torch.no_grad():
        for spec, lbl in va_dl:
            p = torch.sigmoid(model(spec.to(DEVICE))).cpu().numpy()
            probs.extend(p); lbls_all.extend(lbl.numpy())
    auc = roc_auc_score(lbls_all, probs)
    acc = accuracy_score(lbls_all, [p>0.5 for p in probs])
    print(f"  Epoch {epoch}: loss={tr_loss/len(tr_dl):.4f}  val_acc={acc:.4f}  val_auc={auc:.4f}")
    if auc > best_auc:
        best_auc = auc
        torch.save(model.state_dict(), os.path.join(MODEL_DIR,"audio_crnn.pth"))
        print("  ✔ Saved best model")
print(f"\n✅ Done. Best AUC: {best_auc:.4f}")

Epoch 1/20: 100%|██████████| 13/13 [01:26<00:00,  6.63s/it]


  Epoch 1: loss=0.6843  val_acc=0.5321  val_auc=0.5522
  ✔ Saved best model


Epoch 2/20: 100%|██████████| 13/13 [00:48<00:00,  3.73s/it]


  Epoch 2: loss=0.6838  val_acc=0.5064  val_auc=0.5615
  ✔ Saved best model


Epoch 3/20: 100%|██████████| 13/13 [00:46<00:00,  3.55s/it]


...

  Epoch 18: loss=0.6194  val_acc=0.5962  val_auc=0.6463
  ✔ Saved best model


Epoch 19/20: 100%|██████████| 13/13 [00:49<00:00,  3.79s/it]


  Epoch 19: loss=0.6299  val_acc=0.5962  val_auc=0.6468
  ✔ Saved best model


Epoch 20/20: 100%|██████████| 13/13 [00:46<00:00,  3.61s/it]


  Epoch 20: loss=0.6087  val_acc=0.6218  val_auc=0.6457

✅ Done. Best AUC: 0.6468


In [11]:
# ✅ Test Evaluation
model.load_state_dict(torch.load(os.path.join(MODEL_DIR,"audio_crnn.pth"), map_location=DEVICE))
model.eval(); probs, lbls = [], []
with torch.no_grad():
    for spec, lbl in te_dl:
        p = torch.sigmoid(model(spec.to(DEVICE))).cpu().numpy()
        probs.extend(p); lbls.extend(lbl.numpy())
print(f"=== Test Results ===")
print(f"  Accuracy : {accuracy_score(lbls, [p>0.5 for p in probs]):.4f}")
print(f"  AUC      : {roc_auc_score(lbls, probs):.4f}")

=== Test Results ===
  Accuracy : 0.5865
  AUC      : 0.5822
